# Influence of homophily and heterophily on oversmoothing of GAT and TransformerConv

## 1 High homophily graphs

### 1.1 Cora dataset

In [1]:
import torch
import torch_geometric

c:\Users\Mislav.FERIT-PC\Desktop\oversmoothing-GAT-TransformerConv\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
path_cora = "./data/Cora"

In [4]:
cora_dataset = torch_geometric.datasets.Planetoid(
    root=path_cora,
    name="Cora",
    transform=torch_geometric.transforms.NormalizeFeatures()
)

Processing...
Done!


In [5]:
print("Dataset information")
print("===================")
print(f"Number of graphs in dataset: {len(cora_dataset)}")
print(f"Number of features: {cora_dataset.num_features}")
print(f"Number of classes: {cora_dataset.num_classes}")

Dataset information
Number of graphs in dataset: 1
Number of features: 1433
Number of classes: 7


In [6]:
cora_data = cora_dataset[0]
print(cora_data)

Data(x=[2708, 1433], edge_index=[2, 10556], y=[2708], train_mask=[2708], val_mask=[2708], test_mask=[2708])


In [7]:
print("Graph information")
print("==================")
print(f"Number of nodes: {cora_data.num_nodes}")
print(f"Number of edges: {cora_data.num_edges}")
print(f"Has isolated nodes: {cora_data.has_isolated_nodes()}")
print(f"Has self loops: {cora_data.has_self_loops()}")

Graph information
Number of nodes: 2708
Number of edges: 10556
Has isolated nodes: False
Has self loops: False


## 2 GAT network definition

In [8]:
class GATNetwork(torch.nn.Module):
    def __init__(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_heads: int,
        dropout_rate: float = 0.5
    ):
        super().__init__()

        self.layers = []

        input_channels = input_size
        output_channels = hidden_size
        for i in range(number_of_layers - 1):
            self.layers += [
                 torch_geometric.nn.GATv2Conv(in_channels=input_channels, out_channels=output_channels, heads=num_heads),
                 torch.nn.Dropout(p=dropout_rate),
                 torch.nn.ReLU()
            ]
            input_channels = hidden_size * num_heads
        output_channels = output_size
        self.layers.append(torch_geometric.nn.GATv2Conv(in_channels=input_channels, out_channels=output_channels, heads=num_heads, concat=False))
        
        self.layers = torch.nn.ModuleList(self.layers)
        
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for layer in self.layers:
            if isinstance(layer, torch_geometric.nn.conv.MessagePassing):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x

## 3 TransformerConv network definition

In [ ]:
class TransformerConvNetwork(torch.nn.Module):
    def __init__(
        self,
        number_of_layers: int,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_heads: int,
        dropout_rate: float = 0.5
    ):
        super().__init__()

        self.layers = []

        input_channels = input_size
        output_channels = hidden_size
        for i in range(number_of_layers - 1):
            self.layers += [
                torch_geometric.nn.TransformerConv(in_channels=input_channels, out_channels=output_channels, heads=num_heads),
                torch.nn.Dropout(p=dropout_rate),
                torch.nn.ReLU()
            ]
            input_channels = hidden_size * num_heads
        output_channels = output_size
        self.layers.append(torch_geometric.nn.TransformerConv(
            in_channels = input_channels,
            out_channels = output_channels,
            heads = num_heads,
            concat = False
        ))

        self.layers = torch.nn.ModuleList(self.layers)
    
    def forward(self, x: torch.Tensor, edge_index: torch.Tensor):
        for layer in self.layers:
            if isinstance(layer, torch_geometric.nn.MessagePassing):
                x = layer(x, edge_index)
            else:
                x = layer(x)
        return x
